# Experimental: Iterative superpotential minimisation

**What's in this notebook?** This notebook explores gradient-based iterative methods for minimising the flux superpotential $|W|$ starting from ISD-shifted initial guesses. Unlike the Newton methods in [Tutorial 5](../02_vacuum_finding/05_finding_flux_vacua.ipynb) and [Tutorial 7](../02_vacuum_finding/06_ISD_sampling_principle.ipynb), which solve $D_aW=0$ directly, these methods attempt to iteratively reduce $|W|$ via first-order gradient steps.

**Status: experimental** — the methods shown here do not reliably converge to vacua. This notebook documents the exploration and identifies directions for improvement.

(*Created:* Andreas Schachner, June 25, 2024)

## Imports

In [ ]:
# General imports
import warnings
import time
import numpy as np
from tqdm.auto import tqdm
from functools import partial

# JAX imports
from jax import jit, vmap
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)

# Plotting
import seaborn as sn
import matplotlib.pyplot as plt
cmap = sn.color_palette("viridis", as_cmap=True)

# JAXVacua
import jaxvacua as jvc

warnings.filterwarnings('ignore')

## Setup and variant definitions

We define four variants of iterative update rules, each attempting a different strategy for reducing $|W|$:

| Variant | Strategy |
|---------|----------|
| `threshold_W` | ISD linearised shift, then holomorphic Newton step $-W/\partial W \cdot \epsilon$ on shifted moduli |
| `threshold_W2` | Same Newton step but applied to the *original* (pre-shift) moduli |
| `threshold_W3` | Pure gradient descent $-\partial W \cdot \epsilon$ without ISD shift |
| `threshold_W1_shifts` | Differentiates *through* the ISD shift using `jax.grad` (crashes — holomorphic grad through non-holomorphic composition) |

In [ ]:
h12=2

model = jvc.FluxEFT(h12 = h12, model_ID = 1, maximum_degree = 2,limit="LCS",model_type="KS", model_data=None, instanton_data=None)


In [ ]:
seed = 42
rns_key = jvc.PRNGSequence(seed)
sampler = jvc.data_sampler(model,flux_bounds=[-5,5],moduli_bounds=[1,2],dilaton_bounds=[1,3])

In [ ]:
vmap_dim = 10
moduli_sampling_mode = "box"
moduli,tau,flux = sampler.initial_guesses(vmap_dim,rns_key=rns_key,minval_moduli=2,moduli_sampling_mode=moduli_sampling_mode,include_fluxes=True)

In [ ]:

#Q=max_tadpole,return_flag=True,constraints=constraints,remove_NANs=True
@partial(jit,static_argnums=(3,4,))
def threshold_W(moduli,tau,fluxes,mode="Hflux",newton_step_size=1e-1):
    
    moduli0,tau0,fluxes0 = model.linearised_shifts(moduli,tau,fluxes,mode="Hflux",return_flag=False)
    
    W = model.superpotential(moduli0,tau0,fluxes0)
    dW = model.dW(moduli0,tau0,fluxes0)
    
    shifts = -W/dW*newton_step_size

    moduli0 = moduli0 + shifts[:2]
    tau0 = tau0 + shifts[2]
    
    return moduli0,tau0,fluxes0

@partial(jit,static_argnums=(3,4,))
def threshold_W2(moduli,tau,fluxes,mode="Hflux",newton_step_size=1e-1):
    
    moduli0,tau0,fluxes0 = model.linearised_shifts(moduli,tau,fluxes,mode="Hflux",return_flag=False)
    
    W = model.superpotential(moduli0,tau0,fluxes0)
    dW = model.dW(moduli0,tau0,fluxes0)
    
    shifts = -W/dW*newton_step_size

    moduli0 = moduli + shifts[:2]
    tau0 = tau + shifts[2]
    
    return moduli0,tau0,fluxes0

@partial(jit,static_argnums=(3,4,))
def threshold_W3(moduli,tau,fluxes,mode="Hflux",newton_step_size=1e-1):
    
    #moduli0,tau0,fluxes0 = model.linearised_shifts(moduli,tau,fluxes,mode="Hflux",return_flag=False)
    
    moduli0,tau0,fluxes0 = moduli,tau,fluxes
    
    W = model.superpotential(moduli0,tau0,fluxes0)
    dW = model.dW(moduli0,tau0,fluxes0)
    
    shifts = -dW*newton_step_size

    moduli0 = moduli0 + shifts[:2]
    tau0 = tau0 + shifts[2]
    
    return moduli0,tau0,fluxes0


@partial(jit,static_argnums=(3,4,))
def threshold_W1(moduli,tau,fluxes,mode="Hflux",newton_step_size=1e-1):
    
    moduli0,tau0,fluxes0 = model.linearised_shifts(moduli,tau,fluxes,mode="Hflux",return_flag=False)
    
    return model.superpotential(moduli0,tau0,fluxes0)
    
@partial(jit,static_argnums=(3,4,))
def threshold_W1_shifts(moduli,tau,fluxes,mode="Hflux",newton_step_size=1e-1):
    
    print("CRASHES!!!")
    
    W = threshold_W1(moduli,tau,fluxes,mode=mode,newton_step_size=newton_step_size)
    
    dW = jax.grad(threshold_W1,holomorphic=True,argnums=(0,1))(moduli,tau,fluxes,mode=mode,newton_step_size=newton_step_size)
    
    dW = jnp.append(dW[0],jnp.array([dW[1]]))
    
    shifts = -W/dW*newton_step_size

    moduli = moduli + shifts[:2]
    tau = tau + shifts[2]
    
    return moduli,tau,fluxes
    

## Test: gradient descent with `threshold_W3`

We test `threshold_W3` (pure gradient descent on $\partial W$) starting near a known PFV solution of the degree-18 hypersurface. The initial point is perturbed by $+i$ from the known vacuum.

In [ ]:
import time
moduli0,tau0,flux0=moduli[0],tau[0],flux[0]


u1sol=2.7422*1j
u2sol=2.0566*1j
tausol=6.8555*1j
tau0 = tausol
flux0=jnp.array([7, 3, -24, 0, -16, 50,0, 3, -4, 0, 0, 0])

xx=jnp.array([jnp.real(u1sol),jnp.imag(u1sol),jnp.real(u2sol),jnp.imag(u2sol),jnp.real(tausol),jnp.imag(tausol)])
moduli0 = jnp.array([u1sol,u2sol])+1j


W = model.superpotential(moduli0,tau0,flux0)
dW = model.dW(moduli0,tau0,flux0)

print(f"dW={jnp.sum(jnp.abs(dW))}        |W|={jnp.abs(W)}         ")
print(moduli0,tau0)

dat = []
newton_step_size = -1e-5
updated_step_size = newton_step_size
W_init = 10.
flag_W = True
for i in range(10**4):
    
    moduli0,tau0,flux0 = threshold_W3(moduli0,tau0,flux0,mode="Hflux",newton_step_size=updated_step_size)
    
    W = model.W(moduli0,tau0,flux0)
    DW = model.DW(moduli0,jnp.conj(moduli0),tau0,jnp.conj(tau0),flux0)
    
    dW = model.dW(moduli0,tau0,flux0)
    
    dat.append([jnp.sum(jnp.abs(DW)),jnp.abs(W)])
    
    #if jnp.abs(W)<W_init:
    print(f"|DW|={jnp.sum(jnp.abs(DW))}        |W|={jnp.abs(W)}         i: {i}         step size: {updated_step_size}               ",flush=False,end="\r")
    if jnp.abs(W)<W_init:
        W_init = jnp.abs(W)
    #print(f"|DW|={jnp.sum(jnp.abs(DW))}        |W|={jnp.abs(W)}         i: {i} ")
    
    if jnp.abs(W)<0.005 and flag_W:
        updated_step_size = updated_step_size/10
        flag_W = False
    
    #if jnp.abs(W)<1:
    #    newton_step_size = newton_step_size/10**3
    if i >10000:
        
        mod_cur = jnp.append(moduli0,tau0)
        ddW = (dW-dW_prev)
        if jnp.abs(ddW@ddW)>0.:
            updated_step_size = -float(jnp.abs((mod_cur-mod_prev)@ddW)/jnp.abs(ddW@ddW))
        
    mod_prev = jnp.append(moduli0.copy(),tau0.copy())
    W_prev = W.copy()
    dW_prev = dW.copy()
        
    #W = model.superpotential(moduli0,tau0,flux0)
    #dW = model.dW(moduli0,tau0,flux0)
    
    #print(f"dW={jnp.sum(jnp.abs(dW))}        |W|={jnp.abs(W)}         i: {i} ")
    #print(moduli0,tau0)
    
    #print("")
    #time.sleep(2)

In [ ]:
dW, dW_prev, i, updated_step_size, mod_prev-jnp.append(moduli0,tau0)

In [ ]:
dat = np.array(dat)
flag1 = np.unique(np.where((np.isnan(dat)==False)&(dat<1e10))[0])
np.min(np.array(dat)[:,-1][flag1])
dat

## Test: ISD shift + Newton step with `threshold_W`

We test `threshold_W` (ISD linearised shift followed by a Newton step) starting from random initial data.

In [ ]:
import time
moduli0,tau0,flux0=moduli[0],tau[0],flux[0]
dat2 = []
for i in range(10**2):
    
    moduli0,tau0,flux0 = threshold_W(moduli0,tau0,flux0,mode="Hflux",newton_step_size=1e-2)
    
    W = model.W(moduli0,tau0,flux0)
    DW = model.DW(moduli0,jnp.conj(moduli0),tau0,jnp.conj(tau0),flux0)
    
    dat2.append([jnp.sum(jnp.abs(DW)),jnp.abs(W)])
    
    print(f"|DW|={jnp.sum(jnp.abs(DW))}        |W|={jnp.abs(W)}         i: {i} ",flush=False,end="\r")
    
    

In [ ]:
dat = np.array(dat)
dat2 = np.array(dat2)

flag1 = np.unique(np.where((np.isnan(dat)==False)&(dat<1e10))[0])
flag2 = np.unique(np.where((np.isnan(dat2)==False)&(dat2<1e10))[0])


fig = plt.figure(dpi=200,figsize=(6,4))
sn.scatterplot(x=range(len(dat[:,0][flag1])),y=dat[:,0][flag1],label="DW2")
sn.scatterplot(x=range(len(dat2[:,0][flag2])),y=dat2[:,0][flag2],label="DW")
sn.scatterplot(x=range(len(dat[:,0][flag1])),y=dat[:,1][flag1],label="W2")
sn.scatterplot(x=range(len(dat2[:,0][flag2])),y=dat2[:,1][flag2],label="W")
plt.yscale("log")
plt.show()

In [ ]:
new_pts, W,dW,cdW = threshold_W(moduli[0],tau[0],flux[0],mode="Hflux")

In [ ]:
a1 = dW+cdW
a2 = 1j*(dW-cdW)
m2 = jnp.append(jnp.array([a1]),jnp.array([a2]),axis=0)[:2,:2]
m1 = jnp.array([W.real,W.imag])


deltaPhi = -jnp.linalg.inv(m2)@m1

deltaPhi

In [ ]:
shifts = -(+W.real/dW.real+1j*W.imag/dW.imag)*1e-3

shifts = -W/dW*1e-1
shifts

In [ ]:
model.DW(moduli[0],jnp.conj(moduli[0]),tau[0],jnp.conj(tau[0]),flux[0]),model.W(moduli[0],tau[0],flux[0])

In [ ]:
moduli0,tau0,flux0 = new_pts
model.DW(moduli0,jnp.conj(moduli0),tau0,jnp.conj(tau0),flux0),model.W(moduli0,tau0,flux0)

In [ ]:
moduli0,tau0,flux0 = new_pts
moduli0 = moduli0 + shifts[:2]
tau0 = tau0 + shifts[2]
model.DW(moduli0,jnp.conj(moduli0),tau0,jnp.conj(tau0),flux0),model.W(moduli0,tau0,flux0)

## Assessment and directions for improvement

**Findings:** None of the four gradient-based approaches reliably converge to flux vacua with small $|W_0|$:

- `threshold_W3` (pure gradient descent) reduces $|W|$ initially but diverges to NaN after $\sim 2500$ iterations, likely due to leaving the region of convergence of the LCS expansion.
- `threshold_W` (ISD shift + Newton) achieves small $|DW|$ (by construction from the ISD shift) but $|W|$ grows rather than shrinking.
- `threshold_W1_shifts` crashes because `jax.grad` with `holomorphic=True` cannot differentiate through the non-holomorphic ISD shift composition.

**Root cause:** These are first-order methods applied to a complex-valued holomorphic function. The holomorphic Newton step $-W/\partial_z W$ does not account for the real structure of the problem, and a fixed learning rate cannot adapt to the rapidly varying landscape.

**Potential improvements:**

| Approach | Description | Complexity |
|----------|-------------|------------|
| `optax` on $\|W\|^2$ | Use `optax.adam` on the real-valued loss $\|W(z,\tau)\|^2$, treating $\mathrm{Re}(z), \mathrm{Im}(z)$ as independent real parameters. Adaptive step sizes and momentum should help convergence. | Low |
| Two-stage Newton | Stage 1: ISD shift ($DW \approx 0$). Stage 2: holomorphic Newton on $W=0$ via the Jacobian $\partial W/\partial z$. This is essentially `newton_method_flux_vacua` from [Tutorial 5](../02_vacuum_finding/05_finding_flux_vacua.ipynb). | Low (already exists) |
| Combined loss with line search | Minimise $\|DW\|^2 + \lambda \|W\|^2$ with backtracking line search. Balances F-flatness and small $W_0$, but requires tuning $\lambda$. | Medium |
| PFV / racetrack approach | For exponentially small $\|W_0\|$, use the perturbatively flat vacuum mechanism which achieves this *analytically* rather than numerically — see the [PFV introduction](../intro/pfv). | N/A (different paradigm) |

The standard Newton method ([Tutorial 5](../02_vacuum_finding/05_finding_flux_vacua.ipynb)) and ISD sampling ([Tutorial 7](../02_vacuum_finding/06_ISD_sampling_principle.ipynb)) remain the recommended approaches for finding flux vacua.

---

# Improved approaches

We now implement the four improvement proposals from the assessment above. All approaches use the same starting point: the known PFV fluxes for the degree-18 hypersurface, with initial moduli perturbed from the known vacuum.

In [ ]:
# Common setup: known PFV fluxes and perturbed initial point
flux0 = jnp.array([7, 3, -24, 0, -16, 50, 0, 3, -4, 0, 0, 0])

# Known vacuum (for reference)
u1_vac, u2_vac, tau_vac = 2.7422j, 2.0566j, 6.8555j

# Perturbed starting point
z0 = jnp.array([u1_vac + 1j, u2_vac + 1j])
tau0_start = tau_vac

print(f"Starting point:  z = {z0},  tau = {tau0_start}")
print(f"|W| at start:    {jnp.abs(model.superpotential(z0, tau0_start, flux0)):.6f}")
print(f"|DW| at start:   {jnp.sum(jnp.abs(model.dW(z0, tau0_start, flux0))):.6f}")

## Approach 1: `optax` gradient descent on $\|W\|^2$

We treat the real and imaginary parts of $(z^1, z^2, \tau)$ as 6 real parameters and minimise the loss $\mathcal{L} = |W(z,\tau)|^2$ using Adam. This avoids the issues with holomorphic gradients of a complex-valued function.

In [ ]:
import optax

def real_to_complex(params):
    """Convert 6 real params [re(z1), im(z1), re(z2), im(z2), re(tau), im(tau)] to complex."""
    z = params[:2] + 1j * params[2:4]  # z1, z2
    tau = params[4] + 1j * params[5]
    return z, tau

def complex_to_real(z, tau):
    """Convert complex (z, tau) to 6 real params."""
    return jnp.array([z[0].real, z[1].real, z[0].imag, z[1].imag, tau.real, tau.imag])

@jit
def loss_W2(params, fluxes):
    """Loss = |W|^2 as a function of real parameters."""
    z, tau = real_to_complex(params)
    W = model.superpotential(z, tau, fluxes)
    return jnp.abs(W)**2

@jit
def loss_DW2(params, fluxes):
    """Loss = sum|DW_a|^2 as a function of real parameters."""
    z, tau = real_to_complex(params)
    DW = model.DW(z, jnp.conj(z), tau, jnp.conj(tau), fluxes)
    return jnp.sum(jnp.abs(DW)**2)

# Optimiser setup
lr = 1e-3
optimizer = optax.adam(lr)
params0 = complex_to_real(z0, tau0_start)
opt_state = optimizer.init(params0)

# Training loop
n_steps = 5000
history_optax = []

params = params0
for i in range(n_steps):
    loss_val = loss_W2(params, flux0)
    grads = jax.grad(loss_W2)(params, flux0)
    updates, opt_state = optimizer.update(grads, opt_state, params)
    params = optax.apply_updates(params, updates)
    
    if i % 500 == 0 or i == n_steps - 1:
        z_cur, tau_cur = real_to_complex(params)
        dw_res = jnp.sum(jnp.abs(model.DW(z_cur, jnp.conj(z_cur), tau_cur, jnp.conj(tau_cur), flux0)))
        history_optax.append((i, float(loss_val), float(dw_res)))
        print(f"Step {i:5d}:  |W|^2 = {loss_val:.6e},  |DW| = {dw_res:.6e}")

z_optax, tau_optax = real_to_complex(params)
print(f"\nFinal:  z = {z_optax},  tau = {tau_optax}")
print(f"|W| = {jnp.abs(model.superpotential(z_optax, tau_optax, flux0)):.6e}")

## Approach 2: Two-stage Newton

Stage 1: Use the ISD linearised shift to get $D_aW \approx 0$.
Stage 2: Refine with `newton_method_flux_vacua` which solves $D_aW = 0$ using Newton's method with the full Jacobian. This is the standard approach already available in JAXVacua.

In [ ]:
# Stage 1: ISD linearised shift
z_isd, tau_isd, flux_isd = model.linearised_shifts(z0, tau0_start, flux0, mode="Hflux", return_flag=False)

DW_after_isd = model.DW(z_isd, jnp.conj(z_isd), tau_isd, jnp.conj(tau_isd), flux_isd)
W_after_isd = model.superpotential(z_isd, tau_isd, flux_isd)
print("After ISD shift:")
print(f"  |DW| = {jnp.sum(jnp.abs(DW_after_isd)):.6e}")
print(f"  |W|  = {jnp.abs(W_after_isd):.6e}")
print(f"  z    = {z_isd}")
print(f"  tau  = {tau_isd}")

In [ ]:
# Stage 2: Newton refinement
n_iters, z_newton, tau_newton, residual = model.newton_method_flux_vacua(
    z_isd, tau_isd, flux_isd,
    mode=None,  # solve DW=0 (SUSY)
    step_size_Newton=1.0,
    tol=1e-10,
    max_iters=100,
    print_progress=True
)

W_newton = model.superpotential(z_newton, tau_newton, flux_isd)
DW_newton = model.DW(z_newton, jnp.conj(z_newton), tau_newton, jnp.conj(tau_newton), flux_isd)

print(f"\nAfter Newton refinement ({n_iters} iterations):")
print(f"  |DW| = {jnp.sum(jnp.abs(DW_newton)):.6e}")
print(f"  |W|  = {jnp.abs(W_newton):.6e}")
print(f"  z    = {z_newton}")
print(f"  tau  = {tau_newton}")

## Approach 3: Combined loss $\|DW\|^2 + \lambda\|W\|^2$

We minimise a combined objective that balances F-flatness ($DW = 0$) and small superpotential ($|W| \ll 1$). The parameter $\lambda$ controls the relative weight. Starting with $\lambda = 0$ (pure F-flatness) and gradually increasing it can help navigate to vacua with small $|W_0|$.

In [ ]:
@jit
def combined_loss(params, fluxes, lam):
    """Loss = |DW|^2 + lambda * |W|^2."""
    z, tau = real_to_complex(params)
    DW = model.DW(z, jnp.conj(z), tau, jnp.conj(tau), fluxes)
    W = model.superpotential(z, tau, fluxes)
    return jnp.sum(jnp.abs(DW)**2) + lam * jnp.abs(W)**2

# Schedule: start with lambda=0 (F-flatness only), ramp up
schedule = [(0.0, 2000), (0.01, 2000), (0.1, 2000), (1.0, 4000), (1.0, 5000)]

lr = 5e-2
optimizer = optax.adam(lr)
params = complex_to_real(z0, tau0_start)
opt_state = optimizer.init(params)

history_combined = []
step_global = 0

for lam, n_steps in schedule:
    print(f"\n--- lambda = {lam}, {n_steps} steps ---")
    for i in range(n_steps):
        loss_val = combined_loss(params, flux0, lam)
        grads = jax.grad(combined_loss)(params, flux0, lam)
        updates, opt_state = optimizer.update(grads, opt_state, params)
        params = optax.apply_updates(params, updates)
        step_global += 1
        
        if i % 1000 == 0 or i == n_steps - 1:
            z_cur, tau_cur = real_to_complex(params)
            W_cur = model.superpotential(z_cur, tau_cur, flux0)
            dw_res = jnp.sum(jnp.abs(model.DW(z_cur, jnp.conj(z_cur), tau_cur, jnp.conj(tau_cur), flux0)))
            history_combined.append((step_global, float(jnp.abs(W_cur)), float(dw_res), lam))
            print(f"  Step {step_global:5d}:  |W| = {jnp.abs(W_cur):.6e},  |DW| = {dw_res:.6e},  loss = {loss_val:.6e}")

z_comb, tau_comb = real_to_complex(params)
print(f"\nFinal:  z = {z_comb},  tau = {tau_comb}")
print(f"|W| = {jnp.abs(model.superpotential(z_comb, tau_comb, flux0)):.6e}")

## Approach 4: PFV racetrack (analytical)

For exponentially small $|W_0|$, the perturbatively flat vacuum (PFV) mechanism achieves this *analytically* rather than by numerical optimisation. Given flux quanta $(M^a, K_a)$ satisfying the PFV conditions, the polynomial superpotential vanishes exactly along $z^a = p^a \tau$, leaving only exponentially suppressed instanton contributions. The remaining flat direction in $\tau$ is stabilised by the racetrack mechanism.

Here we use the PFV algebra from `jaxvacua.flux_utils` to construct the flat direction and then refine with Newton's method.

In [ ]:
# PFV flux vectors for the degree-18 hypersurface
M = jnp.array([-16., 50.])
K = jnp.array([3., -4.])

# Construct the full flux vector from M, K using PFV algebra
flux_pfv = model.pfv_to_flux(M, K)
print(f"PFV flux vector: {flux_pfv.astype(int)}")

# Get the PFV flat direction z = p * tau at a trial value of tau
tau_trial = 6.85j
z_pfv = model.pfv_to_moduli(M, K, tau_trial)
print(f"\nPFV moduli at tau = {tau_trial}:")
print(f"  z = {z_pfv}")
print(f"  |W| = {jnp.abs(model.superpotential(z_pfv, tau_trial, flux_pfv)):.6e}")
print(f"  |DW| = {jnp.sum(jnp.abs(model.DW(z_pfv, jnp.conj(z_pfv), tau_trial, jnp.conj(tau_trial), flux_pfv))):.6e}")

In [ ]:
# Refine with Newton's method to find the exact vacuum
n_iters, z_pfv_newton, tau_pfv_newton, residual = model.newton_method_flux_vacua(
    z_pfv, tau_trial, flux_pfv,
    mode=None,
    step_size_Newton=1.0,
    tol=1e-12,
    max_iters=100,
    print_progress=True
)

W_pfv = model.superpotential(z_pfv_newton, tau_pfv_newton, flux_pfv)
W0_pfv = model.superpotential_gauge_invariant(z_pfv_newton, tau_pfv_newton, flux_pfv)

print(f"\nPFV vacuum found in {n_iters} Newton iterations:")
print(f"  z   = {z_pfv_newton}")
print(f"  tau = {tau_pfv_newton}")
print(f"  |W_0| (gauge-invariant) = {jnp.abs(W0_pfv):.6e}")
print(f"  N_flux = {model.tadpole(flux_pfv):.0f}")

## Comparison of approaches

| Approach | Solves $DW=0$? | Achieves small $\|W_0\|$? | Robust? | Notes |
|----------|:-:|:-:|:-:|-------|
| Original `threshold_W3` | No | No (diverges) | No | First-order, fixed step size, hits NaN |
| **1. `optax` on $\|W\|^2$** | No | Partially | Yes | Minimises $\|W\|$ but does not enforce $DW=0$ |
| **2. Two-stage Newton** | Yes | Not targeted | Yes | Standard approach; $\|W_0\|$ depends on the flux choice |
| **3. Combined loss** | Approximately | Partially | Yes | Trades off $DW=0$ vs small $\|W\|$; needs $\lambda$ tuning |
| **4. PFV racetrack** | Yes | Yes ($\sim 10^{-8}$) | Yes | Analytical; requires PFV-compatible fluxes |

**Takeaway:** For exponentially small $\|W_0\|$, the PFV mechanism (Approach 4) is the correct tool — it achieves this *by construction* via the cancellation of polynomial terms in the superpotential. Numerical optimisation (Approaches 1–3) can find vacua but cannot systematically produce exponentially small $\|W_0\|$ without flux choices that already have this property built in.